# Activity Classifier: 5-class vs 3-class LOGO-CV

Trains `DecisionTreeClassifier(max_depth=5, min_samples_leaf=5)` on `data/processed/master_dataset.csv` using leave-one-participant-out cross-validation (LOGO-CV), on the 4 features `mean_mag, std_mag, peak_rel, peak_max` (all derived from wrist accelerometer magnitude).

Runs the **same model, same features, same data** against two different target label granularities to compare:
- **5-class**: `label` column (lying/sitting/standing/walking/running)
- **3-class**: `activity_group` column (stationary/walking/running -- lying/sitting/standing grouped)

Each cell below prints results as soon as that step finishes, so the notebook shows the actual computation trace, not just a final summary.

In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib
import os

IN_PATH = "data/processed/master_dataset.csv"
OUT_PATH = "models/activity_classifier.pkl"
FEATURES = ["mean_mag", "std_mag", "peak_rel", "peak_max"]
ACTS_5CLASS = ["lying", "sitting", "standing", "walking", "running"]
ACTS_3CLASS = ["stationary", "walking", "running"]

def make_model():
    return DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=0)

In [2]:
df = pd.read_csv(IN_PATH)
df = df[df["is_transition"] == 0].copy()

X = df[FEATURES].values
groups = df["participant_id"].values

print(f"Clean rows: {len(df)}")
print(f"Participants: {df['participant_id'].nunique()} -> {sorted(df['participant_id'].unique())}")
print(f"Features: {FEATURES}")

Clean rows: 16880
Participants: 18 -> ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18']
Features: ['mean_mag', 'std_mag', 'peak_rel', 'peak_max']


## 5-class LOGO-CV

Target: `label` (lying/sitting/standing/walking/running). Each fold holds out one participant entirely -- trains on everyone else, tests on the held-out person. Printed live, one line per fold as it completes.

In [3]:
y_5class = df["label"].values
logo = LeaveOneGroupOut()

fold_acc_5class = []
all_true_5class, all_pred_5class = [], []
for train_idx, test_idx in logo.split(X, y_5class, groups):
    clf = make_model()
    clf.fit(X[train_idx], y_5class[train_idx])
    pred = clf.predict(X[test_idx])
    held_out = groups[test_idx][0]
    acc = accuracy_score(y_5class[test_idx], pred)
    fold_acc_5class.append((held_out, acc))
    all_true_5class.extend(y_5class[test_idx])
    all_pred_5class.extend(pred)
    print(f"  held-out {held_out}: accuracy = {acc:.3f}")

mean_acc_5class = np.mean([a for _, a in fold_acc_5class])
print(f"\nmean across {len(fold_acc_5class)} folds: {mean_acc_5class:.3f}")

  held-out P01: accuracy = 0.673
  held-out P02: accuracy = 0.570
  held-out P03: accuracy = 0.421
  held-out P04: accuracy = 0.523
  held-out P05: accuracy = 0.202


  held-out P06: accuracy = 0.196
  held-out P07: accuracy = 0.503


  held-out P08: accuracy = 0.506
  held-out P09: accuracy = 0.484
  held-out P10: accuracy = 0.627
  held-out P11: accuracy = 0.691
  held-out P12: accuracy = 0.649


  held-out P13: accuracy = 0.773
  held-out P14: accuracy = 0.673


  held-out P15: accuracy = 0.417
  held-out P16: accuracy = 0.519
  held-out P17: accuracy = 0.700
  held-out P18: accuracy = 0.729

mean across 18 folds: 0.548


In [4]:
cm_5class = confusion_matrix(all_true_5class, all_pred_5class, labels=ACTS_5CLASS)
print("pooled confusion matrix (rows=true, cols=pred):")
print("  " + " ".join(f"{a[:4]:>6s}" for a in ACTS_5CLASS))
for i, a in enumerate(ACTS_5CLASS):
    print(f"  {a[:4]:>4s} " + " ".join(f"{cm_5class[i][j]:6d}" for j in range(len(ACTS_5CLASS))))

print("\nper-class recall:")
for i, a in enumerate(ACTS_5CLASS):
    recall = cm_5class[i][i] / cm_5class[i].sum() if cm_5class[i].sum() > 0 else float("nan")
    print(f"  {a:>10s}: {recall:.3f}")

pooled confusion matrix (rows=true, cols=pred):
    lyin   sitt   stan   walk   runn
  lyin    957    580   1517    283     36
  sitt    881   1583    800     95     15
  stan    666    708   1860    116     23
  walk    256    334    456   2181    147
  runn    131     51    151    406   2647

per-class recall:
       lying: 0.284
     sitting: 0.469
    standing: 0.551
     walking: 0.646
     running: 0.782


## 3-class LOGO-CV (stationary/walking/running)

Same model, same features, same data, same LOGO-CV procedure -- only the target column changes (`activity_group` instead of `label`). This isolates the effect of regrouping the 3 orientation-only classes from any other change.

In [5]:
y_3class = df["activity_group"].values

fold_acc_3class = []
all_true_3class, all_pred_3class = [], []
for train_idx, test_idx in logo.split(X, y_3class, groups):
    clf = make_model()
    clf.fit(X[train_idx], y_3class[train_idx])
    pred = clf.predict(X[test_idx])
    held_out = groups[test_idx][0]
    acc = accuracy_score(y_3class[test_idx], pred)
    fold_acc_3class.append((held_out, acc))
    all_true_3class.extend(y_3class[test_idx])
    all_pred_3class.extend(pred)
    print(f"  held-out {held_out}: accuracy = {acc:.3f}")

mean_acc_3class = np.mean([a for _, a in fold_acc_3class])
print(f"\nmean across {len(fold_acc_3class)} folds: {mean_acc_3class:.3f}")

  held-out P01: accuracy = 0.873
  held-out P02: accuracy = 0.925


  held-out P03: accuracy = 0.708
  held-out P04: accuracy = 0.916
  held-out P05: accuracy = 0.596


  held-out P06: accuracy = 0.559
  held-out P07: accuracy = 0.888


  held-out P08: accuracy = 0.880
  held-out P09: accuracy = 0.852


  held-out P10: accuracy = 0.812
  held-out P11: accuracy = 0.913
  held-out P12: accuracy = 0.975


  held-out P13: accuracy = 0.972
  held-out P14: accuracy = 0.914


  held-out P15: accuracy = 0.991
  held-out P16: accuracy = 0.752


  held-out P17: accuracy = 0.900
  held-out P18: accuracy = 0.927

mean across 18 folds: 0.853


In [6]:
cm_3class = confusion_matrix(all_true_3class, all_pred_3class, labels=ACTS_3CLASS)
print("pooled confusion matrix (rows=true, cols=pred):")
print("  " + " ".join(f"{a[:4]:>6s}" for a in ACTS_3CLASS))
for i, a in enumerate(ACTS_3CLASS):
    print(f"  {a[:4]:>4s} " + " ".join(f"{cm_3class[i][j]:6d}" for j in range(len(ACTS_3CLASS))))

print("\nper-class recall:")
for i, a in enumerate(ACTS_3CLASS):
    recall = cm_3class[i][i] / cm_3class[i].sum() if cm_3class[i].sum() > 0 else float("nan")
    print(f"  {a:>10s}: {recall:.3f}")

pooled confusion matrix (rows=true, cols=pred):
    stat   walk   runn
  stat   9623    426     71
  walk   1118   2134    122
  runn    395    361   2630

per-class recall:
  stationary: 0.951
     walking: 0.632
     running: 0.777


## Summary: same model + features + data, two evaluation scopes

In [7]:
print(f"5-class mean LOGO-CV accuracy: {mean_acc_5class:.3f}")
print(f"3-class mean LOGO-CV accuracy: {mean_acc_3class:.3f}")
print(f"difference: {mean_acc_3class - mean_acc_5class:+.3f}")

5-class mean LOGO-CV accuracy: 0.548
3-class mean LOGO-CV accuracy: 0.853
difference: +0.305


## Export the deployed model

The 5-class model (fit on ALL participants, not just LOGO-CV folds) is what's exported to `models/activity_classifier.pkl` and deployed to `firmware_ble` (see `export_classifier_to_c.py` -> `activity_classifier_5class.h`). The 3-class number above is reported for comparison/context, not a second deployed model.

In [8]:
final_model = make_model()
final_model.fit(X, y_5class)
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
joblib.dump({"model": final_model, "features": FEATURES, "classes": list(final_model.classes_)}, OUT_PATH)
print(f"Final model (trained on all {df['participant_id'].nunique()} participants) saved to {OUT_PATH}")

Final model (trained on all 18 participants) saved to models/activity_classifier.pkl
